In [ ]:
# Healthcare Analytics: Predicting ER Overcrowding

In [ ]:
# ==============================
# Import Required Libraries
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

sns.set(style="whitegrid")

In [ ]:
# ==============================
# Load Dataset
# ==============================

df = pd.read_csv('ER_Dataset.csv')

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# ==============================
# Data Cleaning
# ==============================

# Check missing values
df.isnull().sum()

# Fix Visit_ID if missing
df.loc[5000:, 'Visit_ID'] = range(5001, 8001)

# Rename column for consistency
df.rename(columns={'Equipment_Usage_%': 'Equipment_usage_percent'}, inplace=True)

print("Data cleaning completed.")

In [ ]:
## Exploratory Data Analysis

In [ ]:
# Waiting Time Distribution

plt.figure(figsize=(8,5))
sns.histplot(df['Waiting_Time_Min'], bins=30, kde=True)
plt.title("Distribution of Patient Waiting Time")
plt.show()

In [ ]:
# Overcrowding Frequency

plt.figure(figsize=(6,4))
sns.countplot(x='Overcrowded', data=df)
plt.title("ER Overcrowding Frequency")
plt.show()

In [ ]:
# Waiting Time VS Overcrowding

plt.figure(figsize=(6,4))
sns.boxplot(x='Overcrowded', y='Waiting_Time_Min', data=df)
plt.title("Waiting Time vs Overcrowding")
plt.show()

In [ ]:
# Correlation Heatmap

plt.figure(figsize=(8,6))
sns.heatmap(
    df[['Waiting_Time_Min',
        'Patients_In_ER',
        'Doctor_Available',
        'Equipment_usage_percent',
        'Overcrowded']].corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Hour vs Day Heatmap

pivot = df.pivot_table(
    values='Patients_In_ER',
    index='Day',
    columns='Hour',
    aggfunc='mean'
)

plt.figure(figsize=(10,6))
sns.heatmap(pivot, cmap='YlOrRd')
plt.title("Average ER Patient Load by Day and Hour")
plt.show()

In [ ]:
## Linear Regression: Predicting Waiting Time

In [ ]:
X_lr = df[['Patients_In_ER', 'Doctor_Available', 'Equipment_usage_percent']]
y_lr = df['Waiting_Time_Min']

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

y_pred_lr = lr_model.predict(X_test_lr)

print("R2 Score:", r2_score(y_test_lr, y_pred_lr))
print("Mean Squared Error:", mean_squared_error(y_test_lr, y_pred_lr))

In [ ]:
## Logistic Regression: Predicting Overcrowding

In [ ]:
X = df[['Patients_In_ER',
        'Doctor_Available',
        'Equipment_usage_percent',
        'Waiting_Time_Min']]

y = df['Overcrowded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# ROC Curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
## MySQL Database Integration (Demonstration)

import mysql.connector

try:
    db = mysql.connector.connect(
        host="localhost",
        user="root",
        password="your_password",
        database="er_healthcare"
    )
    print("Connected to MySQL successfully")
    db.close()

except:
    print("Database connection failed")

In [ ]:
## Key Business Insights

- Overcrowding peaks during high patient inflow hours.
- Doctor availability significantly impacts waiting time.
- Equipment utilization correlates with ER congestion.
- Logistic Regression successfully predicts overcrowding scenarios.